# Лабораторная 2. Формирование отчётов в Apache Spark

## Задание

Сформировать отчёт с информацией о 10 наиболее популярных языках программирования по итогам года за период с 2010 по 2020 годы. Отчёт будет отражать динамику изменения популярности языков программирования и представлять собой набор таблиц "топ-10" для каждого года.

Получившийся отчёт сохранить в формате Apache Parquet.

Для выполнения задания вы можете использовать любую комбинацию Spark API: **RDD API**, **Dataset API**, **SQL API**.

## Набор данных

Архивы сайтов **Stack Exchange** доступны по адресу https://archive.org/details/stackexchange.

В папке `data` данного репозитория вам доступны:
- выборка данных `posts_sample.xml` (из stackoverflow.com-Posts.7z),
- файл со списком языков `programming-languages.csv`, собранных с вики-страницы https://en.wikipedia.org/wiki/List_of_programming_languages.

Рекомендуется отлаживать решение на небольшой выборке данных `posts_sample.xml`.

## Подготовка среды в Google Colab

Установка зависимостей

In [1]:
!apt-get update -qq > /dev/null 2>&1
!apt-get install openjdk-11-jdk-headless -y -qq > /dev/null 2>&1
!wget --show-progress -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz -O spark.tgz
!tar xf spark.tgz -C /content/ > /dev/null
!pip install -q findspark

spark.tgz           100%[===================>] 381.90M  7.52MB/s    in 47s     


Настройка переменных окружения

In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages com.databricks:spark-xml_2.12:0.17.0 pyspark-shell"

Инициализация Spark

In [3]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StackOverflowReport") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

spark

## Загрузка данных

programming-languages.csv

In [4]:
!wget -O programming-languages.csv https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/programming-languages.csv

--2026-04-11 14:55:36--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/programming-languages.csv
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/programming-languages.csv [following]
--2026-04-11 14:55:36--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/programming-languages.csv
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 40269 (39K) [text/plain]
Saving to: ‘programming-languages.csv’

programming-languag 100%[===================>]  39.33K  --.-KB/s    in 0.1s    

2026-04-11 14:55:36 (284 KB/s) - ‘programming-languages.csv’ saved [40269/40269]



In [5]:
languages_df = spark.read.format("csv") \
    .option("header", "true") \
    .load("/content/programming-languages.csv") \
    .dropna()

languages_df.show()

+------------+--------------------+
|        name|       wikipedia_url|
+------------+--------------------+
|     A# .NET|https://en.wikipe...|
|  A# (Axiom)|https://en.wikipe...|
|  A-0 System|https://en.wikipe...|
|          A+|https://en.wikipe...|
|         A++|https://en.wikipe...|
|        ABAP|https://en.wikipe...|
|         ABC|https://en.wikipe...|
|   ABC ALGOL|https://en.wikipe...|
|       ABSET|https://en.wikipe...|
|       ABSYS|https://en.wikipe...|
|         ACC|https://en.wikipe...|
|      Accent|https://en.wikipe...|
|    Ace DASL|https://en.wikipe...|
|        ACL2|https://en.wikipe...|
|     ACT-III|https://en.wikipe...|
|     Action!|https://en.wikipe...|
|ActionScript|https://en.wikipe...|
|         Ada|https://en.wikipe...|
|     Adenine|https://en.wikipe...|
|        Agda|https://en.wikipe...|
+------------+--------------------+
only showing top 20 rows



In [6]:
languages_df.count()

699

posts_sample.xml

In [7]:
!wget -O posts_sample.xml https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/posts_sample.xml

--2026-04-11 14:55:53--  https://git.ai.ssau.ru/tk/big_data/raw/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/posts_sample.xml
Resolving git.ai.ssau.ru (git.ai.ssau.ru)... 91.222.131.161
Connecting to git.ai.ssau.ru (git.ai.ssau.ru)|91.222.131.161|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: /tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/posts_sample.xml [following]
--2026-04-11 14:55:53--  https://git.ai.ssau.ru/tk/big_data/raw/commit/352021f75c1d91b3bc10c22731a36bb5aaf1f529/data/posts_sample.xml
Reusing existing connection to git.ai.ssau.ru:443.
HTTP request sent, awaiting response... 200 OK
Length: 74162295 (71M) [text/plain]
Saving to: ‘posts_sample.xml’

posts_sample.xml    100%[===================>]  70.73M  1.07MB/s    in 80s     

2026-04-11 14:57:14 (904 KB/s) - ‘posts_sample.xml’ saved [74162295/74162295]



In [8]:
posts_df = spark.read.format("xml") \
    .option("rowTag", "row") \
    .option("attributePrefix", "_") \
    .load("/content/posts_sample.xml")

posts_df.printSchema()

root
 |-- _AcceptedAnswerId: long (nullable = true)
 |-- _AnswerCount: long (nullable = true)
 |-- _Body: string (nullable = true)
 |-- _ClosedDate: timestamp (nullable = true)
 |-- _CommentCount: long (nullable = true)
 |-- _CommunityOwnedDate: timestamp (nullable = true)
 |-- _CreationDate: timestamp (nullable = true)
 |-- _FavoriteCount: long (nullable = true)
 |-- _Id: long (nullable = true)
 |-- _LastActivityDate: timestamp (nullable = true)
 |-- _LastEditDate: timestamp (nullable = true)
 |-- _LastEditorDisplayName: string (nullable = true)
 |-- _LastEditorUserId: long (nullable = true)
 |-- _OwnerDisplayName: string (nullable = true)
 |-- _OwnerUserId: long (nullable = true)
 |-- _ParentId: long (nullable = true)
 |-- _PostTypeId: long (nullable = true)
 |-- _Score: long (nullable = true)
 |-- _Tags: string (nullable = true)
 |-- _Title: string (nullable = true)
 |-- _ViewCount: long (nullable = true)



In [9]:
posts_df.show(n = 5)

+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount| _CommunityOwnedDate|       _CreationDate|_FavoriteCount|_Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+--------------------+--------------------+--------------+---+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+-

## Предобработка и фильтрация

Список языков в нижнем регистре (для фильтрации)

In [10]:
language_names = [row[0].strip().lower() for row in languages_df.select("name").collect()]
language_names[:10]

['a# .net',
 'a# (axiom)',
 'a-0 system',
 'a+',
 'a++',
 'abap',
 'abc',
 'abc algol',
 'abset',
 'absys']

Фильтр по дате и типу поста (только вопросы)

In [11]:
import pyspark.sql.functions as F

filtered_posts = posts_df.filter(
    F.col("_CreationDate").between("2010-01-01", "2020-12-31") &
    (F.col("_PostTypeId") == "1") &
    F.col("_Tags").isNotNull()
)

filtered_posts.show(n = 5)

+-----------------+------------+--------------------+-----------+-------------+-------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount|_CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|
+-----------------+------------+--------------------+-----------+-------------+-------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+-------------

Извлечение года и парсинг тегов

In [12]:
processed = filtered_posts \
    .withColumn("year", F.substring("_CreationDate", 1, 4)) \
    .withColumn("tags_clean", F.regexp_replace("_Tags", "^<|>$", "")) \
    .withColumn("tags_array", F.split("tags_clean", "><"))

processed.show(n = 5)

+-----------------+------------+--------------------+-----------+-------------+-------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+-----------------+------------+---------+-----------+------+--------------------+--------------------+----------+----+--------------------+--------------------+
|_AcceptedAnswerId|_AnswerCount|               _Body|_ClosedDate|_CommentCount|_CommunityOwnedDate|       _CreationDate|_FavoriteCount|    _Id|   _LastActivityDate|       _LastEditDate|_LastEditorDisplayName|_LastEditorUserId|_OwnerDisplayName|_OwnerUserId|_ParentId|_PostTypeId|_Score|               _Tags|              _Title|_ViewCount|year|          tags_clean|          tags_array|
+-----------------+------------+--------------------+-----------+-------------+-------------------+--------------------+--------------+-------+--------------------+--------------------+----------------------+-----------------+

одна строка = один язык

In [13]:
exploded = processed.select("year", F.explode("tags_array").alias("language"))
exploded.show(n = 5)

+----+------------------+
|year|          language|
+----+------------------+
|2010|               c++|
|2010|character-encoding|
|2010|        sharepoint|
|2010|          infopath|
|2010|            iphone|
+----+------------------+
only showing top 5 rows



Только известные языки программирования

In [14]:
valid_langs = exploded.filter(F.lower(F.col("language")).isin(language_names))
valid_langs.show(n = 5)

+----+--------+
|year|language|
+----+--------+
|2010|    java|
|2010|     php|
|2010|    ruby|
|2010|       c|
|2010|     php|
+----+--------+
only showing top 5 rows



## Формирование топ-10 по годам

Группировка и подсчёт

In [15]:
counts = valid_langs.groupBy("year", "language").count()
counts.show(n = 5)

+----+----------+-----+
|year|  language|count|
+----+----------+-----+
|2013|    erlang|    3|
|2017|typescript|   20|
|2017|       sed|    2|
|2013|javascript|  198|
|2013|        f#|    6|
+----+----------+-----+
only showing top 5 rows



Оконная функция для ранжирования внутри каждого года

In [16]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("year").orderBy(F.desc("count"))

Топ-10 за 2010-2020

In [17]:
top10_report = counts \
    .withColumn("rank", F.row_number().over(window_spec)) \
    .filter(F.col("rank") <= 10) \
    .select(
        F.col("year").alias("Year"),
        F.col("language").alias("Language"),
        F.col("count").alias("Count")
    ) \
    .orderBy("Year", F.desc("Count"))

top10_report.show(100, truncate=False)

+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2010|java       |52   |
|2010|php        |46   |
|2010|javascript |44   |
|2010|python     |26   |
|2010|objective-c|23   |
|2010|c          |20   |
|2010|ruby       |12   |
|2010|delphi     |8    |
|2010|r          |3    |
|2010|applescript|3    |
|2011|php        |102  |
|2011|java       |93   |
|2011|javascript |83   |
|2011|python     |37   |
|2011|objective-c|34   |
|2011|c          |24   |
|2011|ruby       |20   |
|2011|perl       |9    |
|2011|delphi     |8    |
|2011|bash       |7    |
|2012|php        |154  |
|2012|javascript |132  |
|2012|java       |124  |
|2012|python     |69   |
|2012|objective-c|45   |
|2012|c          |27   |
|2012|ruby       |27   |
|2012|bash       |10   |
|2012|r          |9    |
|2012|xpath      |6    |
|2013|javascript |198  |
|2013|php        |198  |
|2013|java       |194  |
|2013|python     |90   |
|2013|objective-c|40   |
|2013|c          |36   |
|2013|ruby       |32   |


## Сохранение в Parquet

In [18]:
output_path = "/content/top_ten_languages.parquet"
top10_report.write.mode("overwrite").parquet(output_path)
print(f"Отчёт сохранён: {output_path}")

Отчёт сохранён: /content/top_ten_languages.parquet


In [19]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("top_ten_languages", "zip", root_dir=output_path)
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
df_restored = spark.read.parquet(output_path)
df_restored.printSchema()

root
 |-- Year: string (nullable = true)
 |-- Language: string (nullable = true)
 |-- Count: long (nullable = true)



In [21]:
df_restored.show(5, truncate=False)

+----+-----------+-----+
|Year|Language   |Count|
+----+-----------+-----+
|2010|java       |52   |
|2010|php        |46   |
|2010|javascript |44   |
|2010|python     |26   |
|2010|objective-c|23   |
+----+-----------+-----+
only showing top 5 rows

